# **1. Object Detection**
Object Detection(객체 탐지)은 이미지나 영상에서 특정 객체의 존재 여부를 확인하고, 해당 객체의 위치를 바운딩 박스(bounding box)로 표시하는 기술입니다. 
<br>이는 컴퓨터 비전에서 중요한 분야로, 이미지 내에서 여러 개의 객체를 동시에 탐지하고 분류할 수 있습니다. 
<br>Object Detection은 주로 딥러닝 기반의 CNN(합성곱 신경망) 모델을 활용하며, 
<br>대표적인 알고리즘으로는 R-CNN 계열(Faster R-CNN, Mask R-CNN), YOLO(You Only Look Once), SSD(Single Shot MultiBox Detector) 등이 있습니다. 
<br>이러한 기술은 자율 주행, 보안 감시, 의료 영상 분석, 증강 현실 등 다양한 분야에서 활용됩니다.

![논문 추천 리스트](./images/논문_추천_리스트.png)

### 1. 객체 탐지의 출력

- 클래스: 갈마, 자동차, 개, 고양이 등
- 바운딩 박스: 객체의 위치와 크기
- confidence score: 해당 예측을 모델이 얼마나 확신하는지 나타내는 점수

> 객체 탐지는 이미지 전체에 하나의 클래스만 붙이는 Image Classification과 다름. 한 이미지 안에 여러 객체가 있어도 각각 따로 찾아내고 점수를 부여할 수 있음

### 2. 대표적안 딥러닝 기반 계역
1. One - Stage
    - 한 번의 네트워크 흐름에서 객체 클래스와 위치를 직접 예측
    - 일반적으로 속도가 빠르며, 실시간 응용에 많이 사용
    - YOLO, SSD, RetinaNet 등

2. Two-Stage
    - 먼저 객체 후보 영역을 만들고, 그 후보를 다시 분류하고 박스를 보정함
    - 일반적으로 정밀한 탐지에 강점이 있음
    - Faster R-CNN, Mask R-CNN 등

> 실무에서는 자율주행, CCTV 분석, 제조 불량 검출, 의료영상 분석 등에서 사용. 현대 모델에서는 "One-Stage"는 조금 부정확하고 "Two-Stage"는 정확하다는 단정을 하면 안됨. 데이터셋, 모델 크기, 학습 방식에 따라 성능 관계는 달라짐

# 2. YOLO
YOLO(You Only Look Once)는 이미지를 격자 단위로 나누어 각 격자에서 객체의 위치(바운딩 박스 좌표)와 클래스를 동시에 예측하는 1단계 객체 탐지(One-Stage Detection) 알고리즘입니다. 
<br>한 번의 신경망 추론으로 전체 이미지의 탐지를 수행하기 때문에 매우 빠르고, 실시간 객체 탐지에 적합하다는 장점이 있습니다. 
<br>YOLO 계열은 v1에서 시작해 v3까지는 Joseph Redmon이 개발했으며, 
<br>이후 v4·v7은 커뮤니티 연구자들, v5·v8·v11은 Ultralytics가 주도적으로 발전시켜 현재까지 이어지고 있습니다. 
<br>이러한 발전 과정을 거치면서 YOLO는 속도와 정확도의 균형을 잡은 대표적인 객체 탐지 모델로 자리 잡아 다양한 산업 현장에서 활용되고 있습니다.

### 1. YOLO 모델 크기별 분류
 - n (nano): 초경량 모델로 모바일 및 임베디드 장치에서 사용하기 적합합니다.
 - s (small): 속도가 빠르면서도 정확도가 준수하여 실시간 추론이 필요한 경우 사용합니다.
 - m (medium): 균형 잡힌 모델로 중간 정도의 성능과 속도를 제공합니다.
 - l (large): 높은 정확도를 요구하는 애플리케이션에서 사용합니다.
 - x (extra-large): 최대 성능을 발휘하지만, 속도가 상대적으로 느립니다.

### 2. YOLO 모델 유형별 분류
 - YOLO Detect: 가장 일반적인 객체 탐지 모델로, 특정 객체의 위치와 크기를 바운딩 박스로 반환합니다.
 - YOLO Segment: 객체 탐지뿐만 아니라 픽셀 단위의 분할(Segmentation)까지 수행하는 모델입니다.
 - YOLO Pose: 사람의 관절 위치를 탐지하여 포즈를 예측하는 모델입니다.
 - YOLO Cls: 이미지를 분류하는 모델입니다.
 - YOLO OBB: 이미지를 회전하여 객체를 탐지하는 모델입니다.

# **PascalVOC 2007**

PascalVOC 2007은 객체 탐지(Object Detection), 분할(Segmentation), 동작 인식(Action Recognition) 등의 다양한 컴퓨터 비전 과제를 위한 벤치마크 데이터셋입니다. 
<br>총 20개의 객체 클래스(예: 사람, 자동차, 개, 고양이 등)를 포함하며, 훈련(train), 검증(val), 테스트(test) 세트로 구성되어 있습니다. 
<br>각 이미지에는 객체의 경계 상자(Bounding Box) 및 해당 클래스 레이블이 주어지며, Mean Average Precision(mAP) 평가 기준을 적용하여 성능을 측정합니다.

In [8]:
import sys
import torch
from pathlib import Path

In [9]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [10]:
PROJECT_ROOT = Path(r"C:\VSGit\11_CV")
print(PROJECT_ROOT)

C:\VSGit\11_CV


In [20]:
DATA_DIR = PROJECT_ROOT / "data"
VOC_ROOT = DATA_DIR / "VOCdevkit" / "VOC2007"
YOLO_ROOT = DATA_DIR / 'Custom_YOLO'

ANNOTATIONS_DIR = VOC_ROOT / "Annotations"
IMAGES_DIR = VOC_ROOT / "JPEGImages"

print("DATA DIR: ", DATA_DIR.resolve())
print("VOC ROOT: ", VOC_ROOT.resolve())
print("YOLO ROOT: ", YOLO_ROOT.resolve())

DATA DIR:  C:\VSGit\11_CV\data
VOC ROOT:  C:\VSGit\11_CV\data\VOCdevkit\VOC2007
YOLO ROOT:  C:\VSGit\11_CV\data\Custom_YOLO


In [12]:
for sub in [
    "images/train2007", "images/val2007", "images/test2007",
    "labels/train2007", "labels/val2007", "labels/test2007"
]:
    (YOLO_ROOT/sub).mkdir(parents=True, exist_ok=True)

# **4. YOLO annotation 기본 구조**

```
<class_id> <x_center> <y_center> <width> <height>
예) 6 0.189370 0.809700 0.378740 0.378258
```
- class_id: 0부터 시작하는 클래스 번호
- x_center, y_center: 박스의 중심점
- width, height: 박스의 너비와 높이
- 좌표는 모두 이미지 크기로 나눈 0~1사이 값

> VOC XML의 (xmin, ymin, xmax, ymax)를 YOLO 형식으로 바꿀 때는 중심점과 크기를 계산한 뒤 이미지 너비와 높이로 정규화해야 함

In [13]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID

{'aeroplane': 0,
 'bicycle': 1,
 'bird': 2,
 'boat': 3,
 'bottle': 4,
 'bus': 5,
 'car': 6,
 'cat': 7,
 'chair': 8,
 'cow': 9,
 'diningtable': 10,
 'dog': 11,
 'horse': 12,
 'motorbike': 13,
 'person': 14,
 'pottedplant': 15,
 'sheep': 16,
 'sofa': 17,
 'train': 18,
 'tvmonitor': 19}

In [15]:
import xml.etree.ElementTree as ET

In [37]:
def voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h):
    x_center = ((xmin + xmax)/2.0) * image_w
    y_center = ((ymin + ymax)/2.0) * image_h
    box_w = (xmax - xmin) * image_w
    box_h = (ymax - ymin) * image_h
    return x_center, y_center, box_w, box_w


In [43]:
def convert_voc_xml(xml_path, label_path):   
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    image_w = float(size.findtext('width'))
    image_h = float(size.findtext('height'))

    lines = []
    for obj in root.findall('object'):
        class_name = obj.findtext('name')
        if class_name not in CLASS_TO_ID:
            continue
        bnd = obj.find('bndbox')
        xmin = float(bnd.findtext('xmin'))
        ymin = float(bnd.findtext('ymin'))
        xmax = float(bnd.findtext('xmax'))
        ymax = float(bnd.findtext('ymax'))

        xmin = max(0.0, min(xmin, image_w))
        xmax = max(0.0, min(xmax, image_w))
        ymin = max(0.0, min(ymin, image_h))
        ymax = max(0.0, min(ymax, image_h))

        if xmax <= xmin or ymax <= ymin:
            continue
        
        x, y, w, h = voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h)
        class_id = CLASS_TO_ID[class_name]
        lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines), encoding="utf-8")

In [ ]:
# 변환 텍스트

# xml_path = Path(r"C:\VSGit\11_CV\data\VOCdevkit\VOC2007\Annotations\000001.xml")
xml_path = Path(ANNOTATIONS_DIR / "000001.xml")

# label_path = Path(DATA_DIR / "000001.text")
label_path = Path(r"C:\VSGit\11_CV\data\000001.text")

convert_voc_xml(xml_path, label_path)

In [53]:
import shutil # 파일 관련 모듈(삭제, 복사 등)

def prepare_split(voc_dir, split_name, output_name):
    split_file = voc_dir / "ImageSets" / "Main" / f"{split_name}.txt"
    image_dir = voc_dir / "JPEGImages"
    annotation_dir = voc_dir / "Annotations"
    out_image_dir = YOLO_ROOT / "images" / output_name
    out_label_dir = YOLO_ROOT / "labels" / output_name

    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok = True)

    image_ids = [x.strip() for x in split_file.read_text(encoding = 'utf-8').splitlines() if x.strip()]
    # print(image_ids)

    for image_id in image_ids:
        src_image = image_dir / f'{image_id}.jpg'
        src_xml = annotation_dir / f'{image_id}.xml'
        shutil.copy2(src_image, out_image_dir / src_image.name)
        convert_voc_xml(src_xml, out_label_dir / f"{image_id}.txt")
    print(f'{output_name}: {len(image_ids):,} images prepared')

In [54]:
prepare_split(VOC_ROOT, 'train', 'train2007')

train2007: 2,501 images prepared


In [55]:
prepare_split(VOC_ROOT, 'val', 'val2007')

val2007: 2,510 images prepared


In [56]:
prepare_split(VOC_ROOT, 'test', 'test2007')

test2007: 4,952 images prepared
